# Running cell2location on NanostringWTA data
In this notebook we we map fetal brain cell types to regions of interest (ROIs) profiled with the NanostringWTA technology, using a version of our cell2location method recommended for probe based spatial transcriptomics data. This notebook should be read after looking at the main cell2location notebooks.

Load the required modules and configure theano settings:

In [ ]:
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl
import os
import pickle
from IPython.display import Image

import cell2location
import scvi

from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

In this NanostringWTA run we profiled 288 regions of interest (ROIs) spanning the full depth of the cortex and at both 19pcw and 14pcw. An example is shown in the image below, together with the cell types we expect in each region:

In [ ]:
Image(filename='../images/GeometricROIs.PNG')

Relevant data can be downloaded like this:

In [ ]:
os.mkdir('./data')
os.system('cd ./data && wget -q https://cell2location.cog.sanger.ac.uk/nanostringWTA/c2l_wta_tutorial_braindata.h5ad')
os.system('cd ./data && wget -q https://cell2location.cog.sanger.ac.uk/nanostringWTA/polioudakis2019_meanExpressionProfiles.csv')

Here we load the anndata object containing the Nanostring data:

In [ ]:
adata_wta = sc.read_h5ad('/nfs/team283/aa16/data/c2l_wta_tutorial_braindata.h5ad')

Here we load an existing matrix of average gene expression profiles for each cell type expected in our nanostringWTA data (taken from the single-cell RNAseq study Polioudakis et al., Neuron, 2019):

In [ ]:
inf_aver = pd.read_csv("data/polioudakis2019_meanExpressionProfiles.csv", index_col=0)

As you can see this anndata object contains probe counts, as well as negative probe counts in adata.obsm['negative_probes'] and nuclei number in adata.obs['nuclei'] and batch info in adata.obs['slide'].

In [ ]:
adata_wta

We need to subset to gene symbols found in both the Nanostring WTA data and reference data:

In [ ]:
subset = [g for g in adata_wta.var_names if g in inf_aver.index]
inf_aver = inf_aver.loc[subset,:]
adata_wta = adata_wta[:,subset]

Now we can create and train the model:

In [ ]:
# prepare anndata for cell2location model
cell2location.models.Cell2location_WTA.setup_anndata(adata=adata_wta, batch_key = 'slide', nuclei_key = 'nuclei',
                                                neg_probes_key = 'negative_probes')

In [ ]:
adata_wta.obsm['negative_probes'].shape[0]

In [ ]:
# create model
mod = cell2location.models.Cell2location_WTA(
    adata_wta, cell_state_df=inf_aver, 
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
) 
mod.view_anndata_setup()

In [ ]:
# train model
mod.train(max_epochs=30000, 
          # train using full data (batch_size=None)
          batch_size=None, 
          # use all data points in training because 
          # we need to estimate cell abundance at all locations
          train_size=1,
          use_gpu=True)

# plot ELBO loss history during training, removing first 1000 epochs from the plot
mod.plot_history(1000)
plt.legend(labels=['full data training']);

In [ ]:
adata_wta.obsm['negative_probes'].shape

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_wta = mod.export_posterior(
    adata_wta, sample_kwargs={'num_samples': 10, 'batch_size': mod.adata.n_obs, 'use_gpu': True}
)

run_name = 'cell2location_WTA_tutorial'
# Save model
mod.save(f"{run_name}", overwrite=True)

# Save anndata object with results
adata_file = f"{run_name}/sp.h5ad"
adata_wta.write(adata_file)

# Load model:
# mod = cell2location.models.Cell2location_WTA.load(f"{run_name}", adata_wta)

An anndata object that has the cell2location results included is saved and can be used for further analysis, as in standard cell2location:

In [ ]:
adata_c2l = sc.read_h5ad('cell2location_WTA_tutorial/sp.h5ad')

In [ ]:
adata_c2l.obsm['means_cell_abundance_w_sf']

Here is an example way for plotting the results:

In [ ]:
colourCode = {'SPN': 'salmon', 'End': 'darkcyan', 'ExDp1': 'deepskyblue', 'ExDp2': 'blue', 'ExM': 'gold', 'ExM-U': 'yellow',
              'ExN': 'darkorange', 'InCGE': 'darkgrey', 'InMGE': 'dimgray', 'IP': 'darkviolet',
              'Mic': 'indianred', 'OPC': 'lightcoral', 'oRG': 'red', 'Per': 'darkgreen',
              'PgG2M': 'rebeccapurple', 'PgS': 'violet', 'vRG': 'lightgreen'}

In [ ]:
subset_19pcw = [adata_c2l.obs['slide'].iloc[i] == '00MU' and
         adata_c2l.obs['Radial_position'].iloc[i] == 2 for i in range(len(adata_c2l.obs['Radial_position']))]

In [ ]:
cell2location.plt.plot_absolute_abundances_1D(adata_c2l, saving = False, roi_subset = subset_19pcw,
                               scaling = 0.33, power = 1, pws = [0,0,100,500,1000,3000], figureSize = (12,8),
                               dimName = 'VCDepth', xlab = 'Cortical Depth', colourCode = colourCode)